In [285]:
# Import required libraries
# pandas -> data manipulation
# numpy -> numerical operations
# ast -> convert string representation of lists into Python objects
import pandas as pd
import numpy as np
import ast

# Load the TMDB movies and credits datasets
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [286]:
# Merge both datasets on the movie title
movies = movies.merge(credits,on ='title')

# Keep only the features required for content-based recommendation
movies = movies[['id','genres','keywords','original_title','overview','cast','crew']]

In [287]:
# Remove rows with missing values
movies = movies.dropna()

In [288]:
# Extract the 'name' field from JSON-like columns
# Example:
# [{"id": 28, "name": "Action"}] -> ["Action"]
def convert(obj):
    y = []
    for i in ast.literal_eval(obj):
        y.append(i["name"])
    return y

        

In [289]:
# Apply the conversion function to genres and keywords
movies['genres']=movies['genres'].apply(convert)
movies['keywords']=movies['keywords'].apply(convert)


In [290]:
# Extract the top 4 cast members for each movie
# Limiting cast size helps reduce noise
def cast_count(obj):
    y = []
    count = 0
    for i in ast.literal_eval(obj):
        if count < 4:
            y.append(i["name"])
            count+=1
        else:
            break
    return y


In [291]:
# Apply cast extraction
movies['cast']=movies['cast'].apply(cast_count)

In [292]:
# Extract the director's name from the crew column
# Directors strongly influence movie similarity
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L 

In [293]:
# Apply director extraction
movies['crew'] = movies['crew'].apply(fetch_director)

In [294]:
# Preprocess text data
# - Split overview into words
# - Remove spaces from multi-word names
# Example: "Sam Worthington" -> "SamWorthington"
movies['overview'] = movies['overview'].apply(lambda x:x.split())
movies['keywords'] = movies['keywords'].apply(lambda x: [i.replace(" ","")for i in x])
movies['cast'] = movies['cast'].apply(lambda x: [i.replace(" ","")for i in x])
movies['crew'] = movies['crew'].apply(lambda x: [i.replace(" ","")for i in x])



In [295]:
# Combine all important features into a single 'tags' column
# This will be used for vectorization
movies["tags"] = movies["overview"]+movies["keywords"]+movies["cast"]+movies["genres"]+movies["crew"]

In [296]:
# Create the final dataframe
# Convert tags list into a single string
# Convert text to lowercase for consistency
final_df = movies[["id","original_title","tags"]].copy()
final_df["tags"]= final_df["tags"].apply(lambda x:" ".join(x))
final_df["tags"] = final_df["tags"].apply(lambda x: x.lower())

In [297]:
# Convert text into numerical vectors using CountVectorizer
# Keep the most frequent 6000 words and remove English stopwords
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features = 6000,stop_words = 'english')

In [298]:
# Transform movie tags into feature vectors
vectors = cv.fit_transform(final_df["tags"]).toarray()

In [299]:
# Import Porter Stemmer for reducing words to their root form
# Example:
# loved, loving, loves -> love
import nltk 
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [300]:
# Function to stem every word in the tags column
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)   
   


In [301]:
# Function to stem every word in the tags column
final_df['tags'] = final_df['tags'].apply(stem)

In [302]:
# Compute cosine similarity between all movie vectors
# Similarity values range from 0 (different) to 1 (identical)
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [303]:
# Recommendation function
# Input: Movie title
# Output: Top 5 most similar movies
def recomm(movie):
    movie_index = final_df[final_df['original_title'] == movie].index[0]
    distance = similarity[movie_index]
    movies_list = sorted(list(enumerate(distance)),reverse = True,key = lambda x:x[1])[1:6]


    for i in movies_list:
        print(final_df.iloc[i[0]].original_title)

